# lab 4 - Wykorzystanie biblioteki Dask w zadaniach Machine Learning.

## Wprowadzenie i wymagania środowiskowe

W zadaniach uczenia maszynowego (Machine Learning) możemy napotkać dwa główne rodzaje ograniczeń skalowalności:
1. **Ograniczenia obliczeniowe (*Compute-bound*):** Dane bez problemu mieszczą się w pamięci RAM pojedynczej maszyny, ale proces trenowania wielu modeli (walidacja krzyżowa, przeszukiwanie siatki hiperparametrów `GridSearchCV` / `RandomizedSearchCV`, zespoły drzew `RandomForest`) trwa zbyt długo. Rozwiązaniem w Dask jest zrównoleglenie obliczeń za pomocą backendu **`joblib.parallel_backend('dask')`**.
2. **Ograniczenia pamięciowe (*Memory-bound*):** Zbiór danych jest większy niż dostępna pamięć RAM pojedynczego komputera. Rozwiązaniem jest:
   * **Uczenie przyrostowe (*Incremental / Out-of-Core learning*)** na partiach (chunkach) danych za pomocą **`dask_ml.wrappers.Incremental`** oraz **`dask_ml.model_selection.IncrementalSearchCV`** (dla modeli wspierających metodę `partial_fit`, np. `SGDClassifier`, `PassiveAggressiveClassifier`, `MLPClassifier`),
   * **Uczenie zespołowe na blokach danych** za pomocą **`dask_ml.ensemble.BlockwiseVotingClassifier`** (dla dowolnych klasyfikatorów `scikit-learn`, które nie posiadają metody `partial_fit`),
   * **Natywny trening rozproszony** na klastrze z użyciem dedykowanych bibliotek, np. **`xgboost.dask`** (`DaskDMatrix`, `xgb.dask.train`, `DaskXGBClassifier`).

> **Wymagane pakiety:** Przed uruchomieniem notatnika upewnij się, że w Twoim środowisku wirtualnym zainstalowane są poniższe biblioteki:
> ```bash
> pip install "dask[complete]" scikit-learn dask-ml xgboost tqdm ipywidgets matplotlib
> ```

## 1. Eksperymenty bez wykorzystania biblioteki Dask.

**Przykład 1**

In [ ]:
from sklearn.datasets import fetch_rcv1, fetch_covtype
import numpy as np

# Pobranie zbioru RCV1 (804 414 próbek, 47 236 cech):
# Dokumentacja sklearn: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_rcv1.html
# rcv1 = fetch_rcv1()
# Dane w oryginale są zwracane jako macierze rzadkie (scipy.sparse.csr_matrix).
# UWAGA: Próba zamiany tak dużej macierzy rzadkiej na tablicę gęstą (.toarray()) wymagałaby
# zaalokowania ponad 283 GB pamięci RAM! W praktyce macierzy rzadkich (gdzie >99% wartości to zera)
# nie konwertuje się do postaci gęstej, ponieważ większość estymatorów scikit-learn oraz XGBoost
# natywnie obsługuje format CSR.

# Mniejszy zbiór tabelaryczny (581 012 wierszy, 54 cechy, 7 klas) do przeprowadzenia przykładu:
cov = fetch_covtype()
X, y = cov.data, cov.target
print(f"Kształt X: {X.shape}, kształt y: {y.shape}, klasy: {np.unique(y)}")

In [ ]:
import dask.array as da

# Przykład konwersji tablic do Dask DataFrame i natychmiastowej materializacji (.compute()) do pandas DataFrame:
data = [X, da.atleast_2d(y).T]
ddf = (
    da.concatenate(data, axis=1)
    .to_dask_dataframe(columns=cov.feature_names + cov.target_names, index=None)
    .compute()
)

In [ ]:
# drop=True zapobiega dodaniu nadmiarowej kolumny 'index'
ddf = ddf.reset_index(drop=True)

In [ ]:
ddf.head()

In [ ]:
# Uwaga na faktyczny typ danych tej ramki po wywołaniu .compute()!
type(ddf)

In [ ]:
import os

DATADIR = './data/'
os.makedirs(DATADIR, exist_ok=True)

In [ ]:
# Ponieważ ddf to pandas DataFrame, zapis tworzy pojedynczy plik Parquet (brak podziału na partycje):
ddf.to_parquet(os.path.join(DATADIR, 'data.parquet'))

# Gdyby ddf był obiektem Dask DataFrame (bez wcześniejszego .compute()), zapis podzielony na partycje wyglądałby tak:
# ddf.to_parquet(DATADIR, name_function=lambda x: f"data-{x}.parquet")

In [ ]:
# Jako że biblioteka XGBoost wymaga, aby wartości dla zmiennej docelowej (y) były indeksowane od 0
# (w zbiorze Covertype klasy mają etykiety 1..7), wykonujemy kodowanie etykiet:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)
print("Zakodowane klasy y:", np.unique(y))

In [ ]:
import time
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm
import xgboost as xgb

n_splits = 5

def fit_and_score(estimator, X_train, X_test, y_train, y_test):
    """Trenuje estymator na zbiorze treningowym i zwraca wynik (accuracy) na obu zbiorach."""
    # Uwaga metodyczna: w produkcyjnym potoku ML zbiór przekazywany do early_stopping
    # powinien być wydzielonym zbiorem walidacyjnym, a nie końcowym zbiorem testowym.
    estimator.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    train_score = estimator.score(X_train, y_train)
    test_score = estimator.score(X_test, y_test)

    return estimator, train_score, test_score


cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=94)

# Standardowa implementacja XGBoost (tree_method="hist") domyślnie wykorzystuje wielowątkowość CPU (OpenMP).
# Na pojedynczej maszynie, gdy dane mieszczą się w pamięci RAM, lokalny trening wielowątkowy jest zazwyczaj
# szybszy niż trening rozproszony z Dask (ze względu na brak narzutu komunikacyjnego między procesami workerów).
# Kiedy jednak dane przekraczają pamięć RAM pojedynczego węzła, podejście klasyczne staje się niemożliwe do uruchomienia.

clf = xgb.XGBClassifier(
    tree_method="hist",
    max_depth=4,
    learning_rate=0.1,
    n_estimators=50,
    early_stopping_rounds=3,
    random_state=94
)

results = {}
start_time = time.perf_counter()

for train, test in tqdm(cv.split(X, y), desc="Training (bez Dask)...", total=n_splits):
    X_train, X_test = X[train], X[test]
    y_train, y_test = y[train], y[test]
    est, train_score, test_score = fit_and_score(
        clone(clf), X_train, X_test, y_train, y_test
    )
    results[est] = (train_score, test_score)

elapsed_no_dask = time.perf_counter() - start_time
print(f"Czas wykonania 5-krotnej walidacji krzyżowej (bez Dask): {elapsed_no_dask:.2f} s")

In [ ]:
for fold_n, (est_model, (tr_score, te_score)) in enumerate(results.items(), start=1):
    print(f"Fold {fold_n}: train_score = {tr_score:.4f}, test_score = {te_score:.4f}")

## 2. Eksperymenty z wykorzystaniem biblioteki Dask.

Zanim przejdziemy do przykładów, uruchommy wspólny lokalny klaster Dask (`Client`), z którego będziemy korzystać w kolejnych sekcjach. Dzięki temu unikniemy wielokrotnego tworzenia i nadpisywania klastrów w tle.

In [ ]:
from dask.distributed import Client

# Bezpieczne zamknięcie ewentualnego wcześniejszego klienta przed utworzeniem nowego:
try:
    client.close()
except NameError:
    pass

client = Client(n_workers=4, threads_per_worker=2, memory_limit="3GB")
client

### 2.0. Zrównoleglenie klasycznych modeli `scikit-learn` przez backend `joblib` (Compute-bound)

Jeżeli dane mieszczą się w pamięci RAM, ale chcemy rozproszyć kosztowny trening wielu drzew lub przeszukiwanie siatki hiperparametrów (`GridSearchCV`, `cross_validate`) na klastrze Dask, najprostszym rozwiązaniem jest użycie menedżera kontekstu `joblib.parallel_backend('dask')`.

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Wybieramy próbkę 50 000 wierszy i uruchamiamy walidację krzyżową lasu losowego na klastrze Dask:
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X), size=50_000, replace=False)
X_sub, y_sub = X[sample_idx], y[sample_idx]

rf = RandomForestClassifier(n_estimators=50, max_depth=12, random_state=42)

with joblib.parallel_backend("dask"):
    scores = cross_val_score(rf, X_sub, y_sub, cv=3, scoring="accuracy")

print(f"Wyniki CV (Joblib + Dask): {scores.round(4)}, średnia: {scores.mean():.4f}")

### 2.1. Natywny rozproszony trening modelu XGBoost z wykorzystaniem Dask

**Przykład 2**

In [ ]:
import dask
import dask.dataframe as dd
import dask.array as da
from dask_ml.model_selection import KFold
from dask_ml.metrics import accuracy_score
import xgboost as xgb
from xgboost import dask as dxgb

In [ ]:
# Konwersja tablic NumPy na Dask Array z jawnym podziałem wierszowym (wszystkie 54 cechy w jednym bloku kolumnowym):
X_da = da.from_array(X, chunks=(120_000, -1))
y_da = da.from_array(y, chunks=(120_000,))
display(X_da)

In [ ]:
# Szczegóły definiowania parametrów: https://xgboost.readthedocs.io/en/latest/parameter.html#learning-task-parameters
# Tu klasyfikacja wieloklasowa (7 klas: 0..6):

params = {
    'objective': 'multi:softmax',
    'tree_method': 'hist',
    'max_depth': 4,
    'eta': 0.1,
    'subsample': 0.5,
    'min_child_weight': 0.5,
    'num_class': 7
}

n_splits = 5
cv = KFold(n_splits=n_splits, shuffle=True, random_state=94)

predictions = {}
fold_scores = {}

start_time = time.perf_counter()

for i, (train_idx, test_idx) in enumerate(cv.split(X_da, y_da), start=1):
    # Wybieramy wszystkie cechy (bez błędnego ucinania ostatniej kolumny :-1)
    # oraz uzgadniamy rozmiary chunków (.compute_chunk_sizes()), czego wymaga DaskDMatrix:
    X_train_fold = X_da[train_idx].compute_chunk_sizes()
    X_test_fold = X_da[test_idx].compute_chunk_sizes()
    y_train_fold = y_da[train_idx].compute_chunk_sizes()
    y_test_fold = y_da[test_idx].compute_chunk_sizes()

    d_train = dxgb.DaskDMatrix(client, X_train_fold, y_train_fold)
    model = xgb.dask.train(client, params=params, dtrain=d_train, num_boost_round=50)

    # xgb.dask.predict zwraca leniwą tablicę Dask Array:
    preds = xgb.dask.predict(client, model, X_test_fold)
    predictions[f'fold_{i}'] = preds

    # Obliczamy wynik accuracy, póki klaster i obiekty Future są aktywne:
    acc = (preds == y_test_fold).mean().compute()
    fold_scores[f'fold_{i}'] = acc
    print(f"Fold {i}: test_score (accuracy) = {acc:.4f}")

elapsed_dask_xgb = time.perf_counter() - start_time
print(f"Czas wykonania 5-krotnej walidacji krzyżowej (XGBoost + Dask): {elapsed_dask_xgb:.2f} s")

### 2.2. Rozbicie całego procesu ML na niezależne zadania

W pierwszej fazie zazwyczaj zajmujemy się procesem ekstrakcji, oczyszczania i wstępnego przetwarzania danych.

W dokumentacji biblioteki Dask znajdziemy cały dział poruszający temat Machine Learning z wykorzystaniem tej biblioteki na różnych etapach procesu:

> **Dokumentacja Dask-ML:** https://ml.dask.org/

Znajdują się tam również przykłady obrazujące, w jakich przypadkach można skorzystać z możliwości Dask w kontekście ML. Należy dokładnie przeczytać uwagi i wskazówki, które się tam znajdują, gdyż nie wszystkie elementy (np. znane biblioteki do tworzenia modeli ML) współpracują z Dask w trybie *„out of the box”* i trzeba wykorzystywać wrappery lub specjalnie przygotowane integracje – np. bibliotekę **Skorch**, która pozwala korzystać z **PyTorch** w sposób kompatybilny ze `scikit-learn`, a co za tym idzie – również z Dask.

#### Etap 1: Ładowanie i wstępne przetwarzanie danych
* W zależności od potrzeb dane mogą być przekazane do etapu treningowego w różnym formacie: `Dask DataFrame`, `Dask Array` lub innym.
* Dostępnych jest wiele formatów przechowywania danych (Parquet, Zarr, HDF5, CSV), a właściwy dobór formatu i bibliotek zależy od dostępnej infrastruktury.
* Dane mogą być również wczytywane i przetwarzane w paczkach (ang. *batches* / *chunks*), co pozwala przetwarzać zbiory przekraczające rozmiar pamięci RAM i rozpraszać je po całym klastrze.

#### Etap 2: Podział danych na potrzeby etapu trenowania modelu
* Jeżeli dysponujemy bardzo dużym zbiorem danych, nie zawsze musimy dzielić cały zbiór na część treningową oraz testową (np. 80/20) i od razu uruchamiać trening na pełnym wolumenie.
* Często lepszym pomysłem jest dobranie reprezentatywnej próbki danych, na której wykonamy wstępną eksplorację i dobór architektury. W zależności od różnorodności zbioru może się okazać, że wynik będzie wystarczająco dobry.
* Pamiętajmy również, że finalna wielkość modelu i ilość zasobów potrzebnych do jego utrzymania w fazie produkcyjnej (inferencji) mogą być kosztowne. Jeżeli mniejszy i szybszy model osiąga porównywalną jakość, w zastosowaniach produkcyjnych zazwyczaj wygrywa.

#### Etap 3: Trening modelu na dużej ilości danych
Faza treningu jest zazwyczaj najbardziej czasochłonna – zarówno ze względu na sam czas obliczeń, jak i iteracyjną naturę eksperymentów (poszukiwanie optymalnych hiperparametrów z użyciem narzędzi takich jak **Optuna**, **MLflow**, `GridSearchCV` / `IncrementalSearchCV` oraz inżynieria cech – *feature engineering*).

Trening modelu na zbiorze przekraczającym pamięć RAM można w `dask-ml` wykonać m.in. na dwa sposoby:
1. **Użycie modelu zespołowego (`BlockwiseVotingClassifier`):** trenuje większą liczbę mniejszych modeli (np. `RandomForestClassifier`, `DecisionTreeClassifier`) niezależnie na kolejnych blokach (partycjach) danych, a podczas predykcji uśrednia ich głosy. Ważne jest, aby podział danych na bloki zachowywał rozkład klas ze zbioru oryginalnego.
   * Zobacz: [`dask_ml.ensemble.BlockwiseVotingClassifier`](https://ml.dask.org/modules/generated/dask_ml.ensemble.BlockwiseVotingClassifier.html)
2. **Uczenie przyrostowe (`Incremental`):** wykorzystuje estymatory z biblioteki `scikit-learn` wspierające metodę **`partial_fit`** (np. `SGDClassifier`, `SGDRegressor`, `PassiveAggressiveClassifier`, `MLPClassifier`, `MultinomialNB`). Wrapper `Incremental` przekazuje kolejne chunki tablicy `Dask Array` sekwencyjnie do metody `partial_fit`, dzięki czemu w pamięci RAM musi mieścić się tylko pojedynczy chunk danych.
   * Zobacz: [`dask_ml.wrappers.Incremental`](https://ml.dask.org/modules/generated/dask_ml.wrappers.Incremental.html)
   * Strategie skalowania w `scikit-learn`: https://scikit-learn.org/stable/computing/scaling_strategies.html
   * Przykłady z dokumentacji: https://examples.dask.org/machine-learning/incremental.html
   * Integracja ze Skorch (PyTorch): https://skorch.readthedocs.io/en/stable/user/parallelism.html

**Przykład 3 (Uczenie przyrostowe – `Incremental` + `StandardScaler` + `SGDClassifier`)**

In [ ]:
# Przykład z użyciem Incremental learning z dokumentacji Dask
import dask
import dask.array as da
from dask_ml.datasets import make_classification
from dask_ml.model_selection import train_test_split
from dask_ml.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from dask_ml.wrappers import Incremental

# Dostosuj wielkość zbioru oraz liczbę/wielkość chunków.
# Wielkość chunka niedobrana prawidłowo do pamięci na workerze może zakończyć przeliczanie całego grafu błędem pamięci.
# Wskazówka: dla n = 10_000_000 i d = 500 tablica X zajmuje ok. 37,25 GB RAM (czas wykonania ok. 15 minut).
# Na potrzeby szybkiej demonstracji w notatniku używamy n = 1_000_000 i d = 100 (ok. 800 MB, czas wykonania ~10 sekund),
# natomiast większy wolumen przekraczający pamięć RAM workerów przetestujesz w Zadaniu 1.

n, d = 1_000_000, 100
X_inc, y_inc = make_classification(
    n_samples=n,
    n_features=d,
    chunks=n // 20,
    flip_y=0.2,
    random_state=42
)
display(X_inc)

X_train, X_test, y_train, y_test = train_test_split(
    X_inc, y_inc, test_size=0.2, shuffle=True, random_state=42
)
display(X_train)

# Rozproszona standaryzacja cech (modele liniowe SGD są wrażliwe na skalę cech):
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Jeżeli dysponujemy wystarczająco dużą ilością pamięci RAM rozproszoną po workerach,
# możemy utrwalić (persist) dane w pamięci RAM workerów w celu przyspieszenia wielokrotnych obliczeń:
# X_train, X_test, y_train, y_test = dask.persist(X_train, X_test, y_train, y_test)

# Wskazówka wydajnościowa: jeśli znamy listę klas z góry (tu klasyfikacja binarna: [0, 1]),
# podajemy ją jawnie zamiast wywoływać da.unique(y_train).compute(), które wymusiłoby
# dodatkowe przeliczenie całego grafu generowania danych jeszcze przed rozpoczęciem treningu!
classes = np.array([0, 1])

est = SGDClassifier(loss='log_loss', penalty='l2', tol=1e-3, random_state=42)
inc = Incremental(est, scoring='accuracy')

inc.fit(X_train, y_train, classes=classes)
test_acc = inc.score(X_test, y_test)
print(f"Dokładność na zbiorze testowym (Incremental SGDClassifier): {test_acc:.4f}")

#### Etap 4: Serializacja modelu i jego wczytywanie
Serializacja modelu jest konieczna, aby przechować wytrenowany estymator w formie trwałej na dysku oraz uniknąć kosztownego ponownego trenowania przed każdym użyciem. W ekosystemie `scikit-learn` oraz `dask-ml` rekomendowanym narzędziem do serializacji modeli zawierających tablice NumPy jest biblioteka **`joblib`** (lub standardowy moduł `pickle`).

In [ ]:
import joblib

def save_model(model, path: str) -> str:
    """Zapisuje wytrenowany model (np. Incremental lub estymator sklearn) na dysku."""
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    joblib.dump(model, path)
    return path

def load_model(path: str):
    """Wczytuje zapisany model z dysku."""
    return joblib.load(path)

# Zapisujemy wytrenowany przed chwilą model z Przykładu 3:
MODEL_PATH = os.path.join(DATADIR, 'incremental_sgd.joblib')
save_model(inc, MODEL_PATH)
print(f"Model zapisany w: {MODEL_PATH}")

**Przykład 4 (Etap 5: Rozproszona inferencja paczkowa – Batch Prediction z Dask Futures)**

W fazie inferencji podajemy do wytrenowanego modelu nowe dane (np. kolejne pliki z danymi klientów lub partiami pomiarów). Zamiast wczytywać model wielokrotnie w każdym zadaniu, możemy wczytać go raz na klastrze za pomocą `client.submit(load_model, ...)` i przekazać obiekt `Future` modelu do funkcji `client.map(predict, filenames, model=model_future)`.

In [ ]:
# Przygotujmy 4 przykładowe pliki paczek (.npy) z pierwszymi 4 blokami danych testowych X_test:
BATCH_DIR = os.path.join(DATADIR, 'batches')
os.makedirs(BATCH_DIR, exist_ok=True)

filenames = []
for idx in range(min(4, X_test.numblocks[0])):
    batch_path = os.path.join(BATCH_DIR, f'batch_{idx}.npy')
    np.save(batch_path, X_test.blocks[idx].compute())
    filenames.append(batch_path)

def predict_file(filename: str, model):
    """Wczytuje paczkę danych z pliku i uruchamia predykcję modelu."""
    data = np.load(filename)
    return model.predict(data)

# 1. Wysyłamy zadanie wczytania modelu na klaster (zwraca Future):
model_future = client.submit(load_model, MODEL_PATH)

# 2. Mapujemy funkcję predykcji na listę plików, przekazując referencję do modelu:
prediction_futures = client.map(predict_file, filenames, model=model_future)

# 3. Odbieramy wyniki ze wszystkich plików (lub strumieniowo za pomocą dask.distributed.as_completed):
batch_results = client.gather(prediction_futures)

for fname, preds in zip(filenames, batch_results):
    print(f"Plik {os.path.basename(fname)} -> liczba predykcji: {len(preds)}, pierwsze 5 klas: {preds[:5]}")

**Przykład 5 (Rozproszona optymalizacja hiperparametrów – `IncrementalSearchCV`)**

In [ ]:
# Przykład pokazujący, jak wykorzystać Dask do rozproszonego poszukiwania optymalnych
# hiperparametrów klasyfikatora wspierającego partial_fit (algorytm Successive Halving).
# Uwaga: używamy osobnych zmiennych X_search, y_search, aby nie nadpisać danych z Przykładu 3.

from dask_ml.model_selection import IncrementalSearchCV
import numpy as np
from dask_ml.datasets import make_classification
from sklearn.linear_model import SGDClassifier

# Generujemy zbiór podzielony na 50 mniejszych chunków (każdy krok partial_fit w IncrementalSearchCV
# trenuje kandydujące modele na kolejnym pojedynczym chunku danych):
X_search, y_search = make_classification(
    n_samples=1_000_000,
    n_features=20,
    chunks=20_000,
    random_state=56
)

model = SGDClassifier(loss='log_loss', tol=1e-3, penalty='elasticnet', random_state=0)

params = {
    'alpha': np.logspace(-4, 0, num=100),
    'l1_ratio': np.linspace(0, 1, num=100),
    'average': [True, False]
}

# n_initial_parameters określa liczbę losowanych na starcie kombinacji hiperparametrów.
# W kolejnych krokach najmniej obiecujące modele są odrzucane, a najlepsze trenują na kolejnych chunkach:
search = IncrementalSearchCV(
    model,
    params,
    random_state=0,
    n_initial_parameters=50,
    patience=5,
    max_iter=50,
    scoring='accuracy'
)
search.fit(X_search, y_search, classes=[0, 1])

In [ ]:
# Najlepszy wynik walidacyjny oraz wybrane hiperparametry:
# Więcej o tym algorytmie:
# https://ml.dask.org/modules/generated/dask_ml.model_selection.IncrementalSearchCV.html

search.best_score_, search.best_params_

### Zadania

**Zadanie 1**  
Uruchom przykład uczenia przyrostowego (*Incremental learning*) z punktu 2.2 (Przykład 3), dobierając parametry tak, aby całkowity rozmiar danych do przeliczenia był **większy niż sumaryczna ilość pamięci RAM przydzielona workerom**.
* **Wskazówka konfiguracyjna:** Aby bezpiecznie przeprowadzić eksperyment na komputerze laboratoryjnym i nie czekać kilkunastu minut na pojedynczy przebieg, skonfiguruj klaster z mniejszym limitem pamięci na workerze, np. `Client(n_workers=4, threads_per_worker=1, memory_limit="1GB")` (sumarycznie **4 GB RAM** dla całego klastra), a następnie wygeneruj zbiór o rozmiarze ok. **5–6 GB** (np. `n_samples=3_000_000, n_features=250`, co daje $3\,000\,000 \times 250 \times 8\text{ B} \approx 6\text{ GB}$).
* Obserwuj Dask Dashboard i sprawdź zachowanie klastra dla różnych wielkości chunka (np. bardzo duży chunk `n // 4`, średni `n // 30`, mały `n // 300`).
* Zobacz, jak wygląda struktura pamięci na workerach i czy przy zbyt dużych chunkach dochodzi do zrzucania pamięci na dysk (*spilling to disk*).
* Wykonaj kilka eksperymentów z pomiarem czasu (`time.perf_counter()`), szukając optymalnego rozmiaru chunka zapewniającego najkrótszy czas wykonania.

**Zadanie 2**  
Dokonaj serializacji wytrenowanego modelu z Zadania 1 na dysk (za pomocą biblioteki `joblib`), a następnie wczytaj go ponownie do nowej zmiennej.
* Uruchom na wczytanym modelu predykcję dla tablicy `X_test` (`y_pred = loaded_model.predict(X_test)`).
* Wyznacz macierz pomyłek (ang. *confusion matrix*) za pomocą `sklearn.metrics.confusion_matrix` oraz wyświetl ją graficznie za pomocą `ConfusionMatrixDisplay` (pamiętaj, aby przed przekazaniem `y_test` i `y_pred` do funkcji ze `scikit-learn` zmaterializować obie tablice Dask Array w jednym przebiegu grafu: `y_true_np, y_pred_np = dask.compute(y_test, y_pred)`).

**Zadanie 3**  
Korzystając z danych treningowych stworzonych w Zadaniu 1 (`X_train, y_train`), uruchom poszukiwanie optymalnych hiperparametrów modelu za pomocą `IncrementalSearchCV` (tak jak w Przykładzie 5).
* **Wskazówka:** Zwróć uwagę na liczbę i wielkość chunków – `IncrementalSearchCV` wykonuje kolejne kroki `partial_fit` na pojedynczych chunkach. Zamiast generować dane od nowa, użyj metody `.rechunk(...)` na `X_train` oraz `y_train`, aby podzielić je na ok. 40–60 mniejszych chunków (lub wybierz reprezentatywny podzbiór wierszy, jeśli zbiór z Zadania 1 jest bardzo duży).
* Porównaj najlepsze wyliczone hiperparametry (`search.best_params_`) oraz dokładność na zbiorze testowym (`search.score(X_test, y_test)`) z modelem bazowym wytrenowanym w Zadaniu 1.

**Zadanie 4 (dodatkowe / dla chętnych)**  
Przetestuj drugie podejście do trenowania modeli na danych przekraczających pamięć RAM opisane w Etapie 3 – klaster modeli głosujących **`dask_ml.ensemble.BlockwiseVotingClassifier`**.
* Utwórz obiekt `BlockwiseVotingClassifier(estimator=DecisionTreeClassifier(max_depth=10, random_state=42), classes=[0, 1])` (zauważ, że drzewo decyzyjne w `scikit-learn` **nie posiada** metody `partial_fit`, więc nie można go użyć z klasą `Incremental`!).
* Wytrenuj go na podzielonych na chunki danych `X_train, y_train` z Zadania 1/3 i porównaj czas treningu oraz dokładność (`score(X_test, y_test)`) z modelem `Incremental(SGDClassifier)`.

In [ ]:
# Miejsce na rozwiązanie Zadania 1


In [ ]:
# Miejsce na rozwiązanie Zadania 2


In [ ]:
# Miejsce na rozwiązanie Zadania 3


In [ ]:
# Miejsce na rozwiązanie Zadania 4 (opcjonalnego)


In [ ]:
# Po zakończeniu wszystkich zadań zamykamy klienta i klaster:
client.close()